# FNO3D pour les équations de Stokes
Ce notebook contient l'implémentation du modèle Fourier Neural Operator 3D (FNO3D) entraîné sur des données de simulation Stokes.

In [ ]:
# ============================================================================
# 0. Installation des dépendances et Imports
# ============================================================================
# !pip install -q numpy matplotlib torch scipy pyvista

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import pyvista as pv
import glob
from scipy.interpolate import griddata
from pathlib import Path
import os
import zipfile
import shutil
import matplotlib.pyplot as plt

print("✅ Dépendances et bibliothèques chargées")

In [ ]:
# ============================================================================
# 1. Définition du Modèle FNO3D
# ============================================================================

class SpectralConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2, modes3):
        super(SpectralConv3d, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        self.modes3 = modes3

        self.scale = (1 / (in_channels * out_channels))
        self.weights1 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3, dtype=torch.complex64))
        self.weights2 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3, dtype=torch.complex64))
        self.weights3 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3, dtype=torch.complex64))
        self.weights4 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3, dtype=torch.complex64))

    def compl_mul3d(self, input, weights):
        return torch.einsum("bixyz,ioxyz->boxyz", input, weights)

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfftn(x, dim=[-3, -2, -1])

        out_ft = torch.zeros([batchsize, self.out_channels, x.size(-3), x.size(-2), x.size(-1)//2 + 1], dtype=torch.complex64, device=x.device)
        
        out_ft[:, :, :self.modes1, :self.modes2, :self.modes3] =             self.compl_mul3d(x_ft[:, :, :self.modes1, :self.modes2, :self.modes3], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2, :self.modes3] =             self.compl_mul3d(x_ft[:, :, -self.modes1:, :self.modes2, :self.modes3], self.weights2)
        out_ft[:, :, :self.modes1, -self.modes2:, :self.modes3] =             self.compl_mul3d(x_ft[:, :, :self.modes1, -self.modes2:, :self.modes3], self.weights3)
        out_ft[:, :, -self.modes1:, -self.modes2:, :self.modes3] =             self.compl_mul3d(x_ft[:, :, -self.modes1:, -self.modes2:, :self.modes3], self.weights4)

        x = torch.fft.irfftn(out_ft, s=(x.size(-3), x.size(-2), x.size(-1)))
        return x

class FNO3d(nn.Module):
    def __init__(self, modes1, modes2, modes3, width):
        super(FNO3d, self).__init__()
        self.modes1 = modes1
        self.modes2 = modes2
        self.modes3 = modes3
        self.width = width
        self.padding = 6
        
        self.fc0 = nn.Linear(3, self.width)
        self.conv0 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)
        self.conv1 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)
        self.conv2 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)
        self.conv3 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)
        self.w0 = nn.Conv3d(self.width, self.width, 1)
        self.w1 = nn.Conv3d(self.width, self.width, 1)
        self.w2 = nn.Conv3d(self.width, self.width, 1)
        self.w3 = nn.Conv3d(self.width, self.width, 1)
        self.fc1 = nn.Linear(self.width, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        grid = self.get_grid(x.shape, x.device)
        x = torch.cat((x, grid), dim=-1)
        x = self.fc0(x)
        x = x.permute(0, 4, 1, 2, 3)
        x = F.pad(x, [0, self.padding, 0, self.padding, 0, self.padding])

        x1 = self.conv0(x)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv1(x)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv2(x)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv3(x)
        x2 = self.w3(x)
        x = x1 + x2

        x = x[..., :-self.padding, :-self.padding, :-self.padding]
        x = x.permute(0, 2, 3, 4, 1)
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        return x

    def get_grid(self, shape, device):
        batchsize, size_x, size_y, size_z = shape[0], shape[1], shape[2], shape[3]
        gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float)
        gridx = gridx.reshape(1, size_x, 1, 1, 1).repeat([batchsize, 1, size_y, size_z, 1])
        gridy = torch.tensor(np.linspace(0, 1, size_y), dtype=torch.float)
        gridy = gridy.reshape(1, 1, size_y, 1, 1).repeat([batchsize, size_x, 1, size_z, 1])
        return torch.cat((gridx, gridy), dim=-1).to(device)

print("✅ Modèle FNO3D défini")

In [ ]:
# ============================================================================
# 2. Chargement du Modèle Entraîné et des Statistiques
# ============================================================================

# Chemins vers les fichiers (relatifs au dossier models/fno-stokes)
MODEL_PATH = "fno3d_stokes.pth"
STATS_PATH = "normalization_stats_stokes.npz"

# Paramètres du modèle (doivent correspondre à l'entraînement)
MODES = 8
WIDTH = 20

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialisation et chargement
model = FNO3d(MODES, MODES, MODES, WIDTH).to(device)

if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    print(f"✅ Modèle chargé depuis {MODEL_PATH}")
else:
    print(f"❌ Erreur : Fichier {MODEL_PATH} non trouvé.")

# Chargement des statistiques de normalisation
if os.path.exists(STATS_PATH):
    stats = np.load(STATS_PATH)
    x_mean = stats['x_mean']
    x_std = stats['x_std']
    y_mean = stats['y_mean']
    y_std = stats['y_std']
    print(f"✅ Statistiques de normalisation chargées depuis {STATS_PATH}")
else:
    print(f"❌ Erreur : Fichier {STATS_PATH} non trouvé.")

In [ ]:
# ============================================================================
# 3. Fonctions de Prédiction et Visualisation
# ============================================================================

def predict_stokes(input_data):
    """
    input_data: numpy array de forme (nx, ny, nz, 1) représentant le marker/géométrie
    """
    # Normalisation de l'entrée
    input_norm = (input_data - x_mean) / (x_std + 1e-8)
    
    # Conversion en tensor
    input_tensor = torch.tensor(input_norm, dtype=torch.float).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output_norm = model(input_tensor)
    
    # Dénormalisation de la sortie
    output_data = output_norm.cpu().numpy().squeeze() * y_std + y_mean
    return output_data

print("✅ Fonctions de prédiction prêtes")

## Exemple d'utilisation
Vous pouvez maintenant utiliser `predict_stokes` pour obtenir le champ de pression à partir d'une géométrie donnée.